# 01 · Parse & inspect
Step 1 of the build plan: extract text and tables from all 9 filings and sanity-check the table extraction before any chunking.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
from IPython.display import Markdown, display
from finsight.config import list_documents, FISCAL_YEAR_END
from finsight.ingest.parsers import parse_document
docs = list_documents()
pd.DataFrame([{'doc_id': d.doc_id, 'company': d.company, 'fiscal_year': d.fiscal_year, 'type': d.doc_type, 'file': d.path.name} for d in docs])

,doc_id,company,fiscal_year,type,file
0,alphabet_2023,Alphabet,2023,10-K,goog-10-k-2023-final.pdf
1,alphabet_2024,Alphabet,2024,10-K,goog-10-k-2024.pdf
2,alphabet_2025,Alphabet,2025,10-K,GOOG-10-K-2025.pdf
3,amazon_2023,Amazon,2023,annual_report,Amazon-com-Inc-2023-Annual-Report.pdf
4,amazon_2024,Amazon,2024,annual_report,Amazon-2024-Annual-Report.pdf
5,amazon_2025,Amazon,2025,annual_report,Amazon-2025-Annual-Report.pdf
6,microsoft_2023,Microsoft,2023,annual_report,2023_Annual_Report.docx
7,microsoft_2024,Microsoft,2024,annual_report,2024_Annual_Report.docx
8,microsoft_2025,Microsoft,2025,annual_report,2025_AnnualReport.docx


In [2]:
import finsight.config as c; print(c.__file__, c.DATA_DIR)


c:\Users\senpr\Documents\finsight-ai\src\finsight\config.py C:\Users\senpr\Documents\finsight-ai\DataSource


## Parse every document

In [3]:
parsed = {d.doc_id: parse_document(d) for d in docs}
rows = []
for d in docs:
    els = parsed[d.doc_id]
    tb = [e for e in els if e.kind == 'table']
    rows.append({'doc_id': d.doc_id, 'text_blocks': sum(e.kind == 'text' for e in els), 'text_chars': sum(len(e.text) for e in els if e.kind == 'text'),
                 'tables': len(tb), 'median_rows': pd.Series([len(t.rows) for t in tb]).median(), 'max_rows': max(len(t.rows) for t in tb)})
summary = pd.DataFrame(rows).set_index('doc_id'); summary

,text_blocks,text_chars,tables,median_rows,max_rows
doc_id,,,,,
alphabet_2023,140,322689,43,7.0,40
alphabet_2024,139,333116,40,7.0,22
alphabet_2025,147,322353,48,7.0,41
amazon_2023,126,310271,35,8.0,36
amazon_2024,123,303332,33,9.0,36
amazon_2025,124,305838,33,9.0,35
microsoft_2023,68,242543,68,8.0,41
microsoft_2024,70,240664,70,8.0,42
microsoft_2025,63,195465,63,9.0,42


## Eyeball tables
Pick a document and a table index. Check that headers, units and numbers survived.

In [4]:
def show(doc_id, i):
    tb = [e for e in parsed[doc_id] if e.kind == 'table']
    t = tb[i]
    print(f'{doc_id} · table {i}/{len(tb)} · page {t.page} · {len(t.rows)} rows')
    display(Markdown(t.text[:3000]))
show('amazon_2024', 5)

amazon_2024 · table 5/33 · page 40 · 6 rows


|  | 2023 | 2024 |
|---|---|---|
| Net cash provided by (used in) operating activities | $84,946 | $115,877 |
| Purchases of property and equipment, net of proceeds from sales and incentives | (48,133) | (77,658) |
| Free cash flow | $36,813 | $38,219 |
| Net cash provided by (used in) investing activities | $(49,833) | $(94,342) |
| Net cash provided by (used in) financing activities | $(15,879) | $(11,812) |

In [5]:
show('alphabet_2024', 10)

alphabet_2024 · table 10/40 · page 65 · 11 rows


|  | 2022 | 2023 | 2024 |
|---|---|---|---|
| Google Search & other | $162,450 | $175,033 | $198,084 |
| YouTube ads | 29,243 | 31,510 | 36,147 |
| Google Network | 32,780 | 31,312 | 30,359 |
| Google advertising | 224,473 | 237,855 | 264,590 |
| Google subscriptions, platforms, and devices | 29,055 | 34,688 | 40,340 |
| Google Services total | 253,528 | 272,543 | 304,930 |
| Google Cloud | 26,280 | 33,088 | 43,229 |
| Other Bets | 1,068 | 1,527 | 1,648 |
| Hedging gains (losses) | 1,960 | 236 | 211 |
| Total revenues | $282,836 | $307,394 | $350,018 |

In [6]:
show('microsoft_2024', 10)

microsoft_2024 · table 10/70 · page None · 8 rows


| (In millions) |  |
|---|---|
| Three Months Ending |  |
| September 30, 2024 | $ 22,529 |
| December 31, 2024 | 17,664 |
| March 31, 2025 | 12,076 |
| June 30, 2025 | 5,313 |
| Thereafter | 2,602 |
| Total | $ 60,184 |

## Find a known table
Search for a table containing a label we expect, to confirm the key financial statements were captured as tables.

In [7]:
def find(doc_id, needle):
    return [(i, t.page, len(t.rows)) for i, t in enumerate(e for e in parsed[doc_id] if e.kind == 'table') if needle.lower() in t.text.lower()]
for doc, needle in [('amazon_2024', 'Total net sales'), ('alphabet_2024', 'Total revenues'), ('microsoft_2024', 'Intelligent Cloud')]:
    print(doc, needle, find(doc, needle)[:8])

amazon_2024 Total net sales [(10, 49, 26)]
alphabet_2024 Total revenues [(2, 37, 11), (5, 39, 7), (10, 65, 11), (11, 65, 6)]
microsoft_2024 Intelligent Cloud [(4, None, 11), (41, None, 5), (52, None, 6), (66, None, 12)]


## Verify metadata
Confirm the fiscal year inferred from each file name by looking at the opening text.

In [8]:
for d in docs:
    first = ' '.join(e.text for e in parsed[d.doc_id][:3] if e.kind == 'text')[:200].replace(chr(10), ' ')
    print(d.doc_id, '|', FISCAL_YEAR_END[d.company], '|', first)

alphabet_2023 | December 31 | UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 ___________________________________________ FORM 10-K ___________________________________________ (Mark One) ☒ ANNUAL REPORT PUR
alphabet_2024 | December 31 | UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 ___________________________________________ FORM 10-K ___________________________________________ (Mark One) ☒ ANNUAL REPORT PUR
alphabet_2025 | December 31 | UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 ___________________________________________ FORM 10-K ___________________________________________ (Mark One) ☒ ANNUAL REPORT PUR
amazon_2023 | December 31 | 3 2 0 2 A N N U A L R E P O R T Dear Shareholders: Last year at this time, I shared my enthusiasm and optimism for Amazon’s future. Today, I have even more. The reasons are many, but start with the pr
amazon_2024 | December 31 | 4 2 0 2 A N N U A L R E P O R T Dear Shareholders:

## Notes / issues found
_(fill in after reviewing the samples above)_